In [12]:
from pathlib import Path
import pandas as pd
import re

SG_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/datasets/singapore_DB")
RESULTS_DIR = Path("/content/drive/MyDrive/Colab Notebooks/BM/sleep_sg")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

schedule = pd.read_csv(SG_ROOT / "ScheduleInfo.csv")
schedule.columns = [c.strip() for c in schedule.columns]

def normalize_subject_id(x):
    x = str(x).strip().upper()
    digits = re.sub(r"\D", "", x)

    if x.startswith("NUS") and len(digits) == 4:
        return f"NUS{digits}"
    if len(digits) <= 3:
        return f"NUS2{int(digits):03d}"
    if len(digits) == 4 and digits.startswith("2"):
        return f"NUS{digits}"
    return x

def normalize_abpm_on(x):
    return str(x).strip().lower() in ["1", "true", "yes", "y"]

schedule["subject_id"] = schedule["SubID"].apply(normalize_subject_id)
schedule["subject_number"] = schedule["subject_id"].str.replace("NUS", "", regex=False)
schedule["Night"] = pd.to_numeric(schedule["Night"], errors="coerce").astype("Int64")
schedule["ABPM_on_bool"] = schedule["ABPM_on"].apply(normalize_abpm_on)

# ==========================
# Night-level check
# ==========================
rows = []

for _, row in schedule.iterrows():
    if pd.isna(row["Night"]):
        continue

    subject_id = row["subject_id"]
    night = int(row["Night"])
    night_label = f"Night{night}"

    night_dir = SG_ROOT / f"{subject_id}_night{night:02d}"
    bp_files = sorted(night_dir.glob("*_bp.csv")) if night_dir.exists() else []

    rows.append({
        "subject_number": row["subject_number"],
        "subject_id": subject_id,
        "night": night_label,
        "schedule_abpm_on": bool(row["ABPM_on_bool"]),
        "actual_bp_file_exists": len(bp_files) > 0,
        "bp_file_name": bp_files[0].name if len(bp_files) > 0 else None,
    })

night_check = pd.DataFrame(rows)

# ==========================
# Schedule에서 ABPM_on=True + 실제 bp.csv 존재하는 night만
# ==========================
abpm_subjects_valid = (
    night_check[
        (night_check["schedule_abpm_on"]) &
        (night_check["actual_bp_file_exists"])
    ]
    .sort_values("subject_id")
    .reset_index(drop=True)
)

print(f"ABPM_on=True and actual bp.csv exists: {len(abpm_subjects_valid)}")

pd.set_option("display.max_rows", None)

display(
    abpm_subjects_valid[
        ["subject_number", "subject_id", "night", "bp_file_name"]
    ]
)

subject_numbers = (
    abpm_subjects_valid["subject_number"]
    .astype(int)
    .sort_values()
    .tolist()
)

print("\nABPM subject numbers:")
print(subject_numbers)

# ==========================
# Subject-level summary
# ==========================
subject_rows = []

for subject_id, g in night_check.groupby("subject_id"):
    subject_number = g["subject_number"].iloc[0]

    night1 = g[g["night"] == "Night1"]
    night2 = g[g["night"] == "Night2"]

    n1_abpm = (
        bool(night1["schedule_abpm_on"].iloc[0])
        and bool(night1["actual_bp_file_exists"].iloc[0])
    ) if len(night1) else False

    n2_abpm = (
        bool(night2["schedule_abpm_on"].iloc[0])
        and bool(night2["actual_bp_file_exists"].iloc[0])
    ) if len(night2) else False

    if n1_abpm and n2_abpm:
        group = "both_nights_abpm"
    elif n1_abpm or n2_abpm:
        group = "one_night_abpm"
    else:
        group = "no_abpm"

    subject_rows.append({
        "subject_number": subject_number,
        "subject_id": subject_id,
        "night1_abpm_valid": n1_abpm,
        "night2_abpm_valid": n2_abpm,
        "group": group,
    })

subject_check = pd.DataFrame(subject_rows)

summary_df = pd.DataFrame({
    "Group": [
        "Total Subjects",
        "Both Nights ABPM",
        "One Night ABPM",
        "No ABPM"
    ],
    "Count": [
        subject_check["subject_id"].nunique(),
        (subject_check["group"] == "both_nights_abpm").sum(),
        (subject_check["group"] == "one_night_abpm").sum(),
        (subject_check["group"] == "no_abpm").sum(),
    ]
})

print("\n" + "=" * 50)
print("ABPM SUBJECT SUMMARY")
print("=" * 50)
display(summary_df)

print("\nNight1/Night2 count:")
print("Night1 ABPM:", subject_check["night1_abpm_valid"].sum())
print("Night2 ABPM:", subject_check["night2_abpm_valid"].sum())

# 저장
night_check.to_csv(RESULTS_DIR / "abpm_night_check_schedule_and_file.csv", index=False)
subject_check.to_csv(RESULTS_DIR / "abpm_subject_summary_schedule_and_file.csv", index=False)
abpm_subjects_valid.to_csv(RESULTS_DIR / "valid_abpm_subjects.csv", index=False)

print("\nSaved:")
print(RESULTS_DIR / "abpm_night_check_schedule_and_file.csv")
print(RESULTS_DIR / "abpm_subject_summary_schedule_and_file.csv")
print(RESULTS_DIR / "valid_abpm_subjects.csv")

ABPM_on=True and actual bp.csv exists: 76


,subject_number,subject_id,night,bp_file_name
0,2006,NUS2006,Night1,NUS2006_night01_090421_bp.csv
1,2007,NUS2007,Night2,NUS2007_night02_092821_bp.csv
2,2008,NUS2008,Night2,NUS2008_night02_090521_bp.csv
3,2010,NUS2010,Night1,NUS2010_night01_091421_bp.csv
4,2011,NUS2011,Night2,NUS2011_night02_101021_bp.csv
5,2012,NUS2012,Night1,NUS2012_night01_091121_bp.csv
6,2013,NUS2013,Night2,NUS2013_night02_101021_bp.csv
7,2015,NUS2015,Night1,NUS2015_night01_090721_bp.csv
8,2016,NUS2016,Night2,NUS2016_night02_091021_bp.csv
9,2017,NUS2017,Night1,NUS2017_night01_102821_bp.csv



ABPM subject numbers:
[2006, 2007, 2008, 2010, 2011, 2012, 2013, 2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023, 2025, 2026, 2027, 2028, 2029, 2030, 2032, 2033, 2034, 2035, 2036, 2037, 2038, 2040, 2042, 2043, 2044, 2046, 2047, 2048, 2049, 2050, 2051, 2054, 2055, 2056, 2057, 2058, 2059, 2060, 2065, 2066, 2067, 2068, 2069, 2070, 2071, 2073, 2074, 2076, 2077, 2078, 2079, 2080, 2081, 2082, 2083, 2084, 2086, 2087, 2088, 2089, 2090, 2091, 2092, 2093, 2094, 2095, 2096, 2097, 2098]

ABPM SUBJECT SUMMARY


,Group,Count
0,Total Subjects,106
1,Both Nights ABPM,0
2,One Night ABPM,76
3,No ABPM,30



Night1/Night2 count:
Night1 ABPM: 39
Night2 ABPM: 37

Saved:
/content/drive/MyDrive/Colab Notebooks/BM/sleep_sg/abpm_night_check_schedule_and_file.csv
/content/drive/MyDrive/Colab Notebooks/BM/sleep_sg/abpm_subject_summary_schedule_and_file.csv
/content/drive/MyDrive/Colab Notebooks/BM/sleep_sg/valid_abpm_subjects.csv
